# Homework 8: Inverse Probability Weighting and Mahalanobis Matching

Put `homework_8.1.csv` and `homework_8.2.csv` in the same folder as this notebook.

- `homework_8.1.csv` has columns **X, Y, Z**: X is the binary treatment, Y the outcome, Z the confounder.
- `homework_8.2.csv` has columns **X, Y, Z1, Z2**: the same, with two confounders.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from scipy.spatial.distance import mahalanobis

## Question 1: ATE with inverse probability weighting (IPW)

**The problem.** Z affects both who gets treated (X) and the outcome (Y). If we just compared the
average Y of treated vs untreated, part of the difference would come from Z, not from X.

**Step 1: propensity score.** The propensity score P is each row's probability of being treated,
given its Z: `P = Pr(X = 1 | Z)`. We estimate it with logistic regression (Z predicts X), then use
`predict_proba(...)[:, 1]`, the second column, which is the probability of class X = 1.

**Step 2: weights.**
- Treated rows get weight `1 / P`.
- Untreated rows get weight `1 / (1 - P)`.

**Intuition for the weights.** A treated row with a *low* P is rare: people like it usually don't get
treated. It gets a big weight so it can "stand in" for all the similar people who weren't treated.
A treated row with a *high* P is common among the treated, so it gets a small weight. After weighting,
the treated and untreated groups each look like the *whole* population in terms of Z, as if treatment
had been assigned at random. The confounding from Z is removed.

**Step 3: ATE.** Take the weighted average of Y in each group and subtract:

`ATE = weighted mean of Y (treated) - weighted mean of Y (untreated)`

**Choices made in the code (useful for your reflection):**
- I used sklearn's default `LogisticRegression()`. It has a small amount of regularization, which barely
  affects a single predictor with lots of data.
- I used weighted means (`np.average` with weights), which divides by the sum of weights in each group.
  The alternative "Horvitz-Thompson" form divides by N instead. Both are printed; they should be close.
- Z is passed as `df[["Z"]]` (double brackets) because sklearn needs a 2-D table of features, not a 1-D column.

In [2]:
df1 = pd.read_csv("homework_8.1.csv")

# Step 1: estimate propensity scores, P = Pr(X = 1 | Z)
model = LogisticRegression()
model.fit(df1[["Z"]], df1["X"])
df1["P"] = model.predict_proba(df1[["Z"]])[:, 1]   # column 1 = probability that X = 1

# Step 2: inverse probability weights
df1["weight"] = np.where(df1["X"] == 1, 1 / df1["P"], 1 / (1 - df1["P"]))

# Step 3: weighted mean of Y in each group, then the difference
treated = df1[df1["X"] == 1]
untreated = df1[df1["X"] == 0]
y1 = np.average(treated["Y"], weights=treated["weight"])
y0 = np.average(untreated["Y"], weights=untreated["weight"])
ate = y1 - y0

# For comparison: the naive difference with no weighting (biased by Z)
naive = treated["Y"].mean() - untreated["Y"].mean()

# Horvitz-Thompson version: divide by N instead of the sum of weights
X, Y, P = df1["X"], df1["Y"], df1["P"]
ate_ht = np.mean(X * Y / P) - np.mean((1 - X) * Y / (1 - P))

print(f"Naive difference (biased):   {naive:.3f}")
print(f"IPW ATE (weighted means):    {ate:.3f}")
print(f"IPW ATE (Horvitz-Thompson):  {ate_ht:.3f}")

Naive difference (biased):   3.037
IPW ATE (weighted means):    2.274
IPW ATE (Horvitz-Thompson):  2.239


## Question 2: Propensity scores of the first three rows

These are just the P values computed in Question 1 for rows 0, 1 and 2.

**Sanity check:** rows with a larger Z should have a higher chance of being treated if Z pushes people
toward treatment, so their P will be higher.

In [3]:
print("First three propensity scores:", df1["P"].head(3).round(2).tolist())
print(df1[["X", "Z", "P"]].head(3))

First three propensity scores: [0.84, 0.58, 0.71]
   X         Z         P
0  1  1.764052  0.840114
1  0  0.400157  0.584646
2  0  0.978738  0.711082


## Question 3: ATE with Mahalanobis matching

**The idea.** For each treated row, find the untreated row that looks most similar on (Z1, Z2), and treat
its Y as the "what would have happened without treatment" (the counterfactual). The effect for that row is
`Y_treated - Y_matched_untreated`. Average these over all treated rows.

**Why Mahalanobis distance instead of ordinary (Euclidean) distance?**
Euclidean distance treats every variable the same. That goes wrong when:
- the variables are on different **scales** (a gap of 1 in a variable with SD 10 is tiny; in one with SD 0.1 it's huge), or
- the variables are **correlated** (if Z1 and Z2 move together, a point that breaks that pattern is more unusual than it looks).

Mahalanobis distance fixes both by using the **inverse covariance matrix** (VI). It effectively rescales each
variable by its spread and removes their correlation before measuring distance:

`d(u, v) = sqrt( (u - v)ᵀ · VI · (u - v) )`

**Building VI, as the instructions say:** stack all Z1 and all Z2 into a 2 x N matrix, compute its 2 x 2
covariance with `np.cov`, then invert it with `np.linalg.inv`. (`np.cov` treats each **row** as a variable,
which is why we stack them as rows.)

**With replacement:** the same untreated row can be the best match for several treated rows. We simply take the
minimum distance for each treated row and never remove anything from the pool.

**Note:** because we only match the treated rows, this average is technically the effect on the treated (ATT).
The homework calls it the ATE, and this is the number it asks for.

In [7]:
df2 = pd.read_csv("homework_8.2.csv")

# Inverse covariance matrix from ALL rows: 2 x N matrix -> 2 x 2 covariance -> invert
Z = np.vstack([df2["Z1"], df2["Z2"]])   # shape (2, N): each row is one variable
VI = np.linalg.inv(np.cov(Z))           # shape (2, 2)

treated2 = df2[df2["X"] == 1].reset_index(drop=True)
untreated2 = df2[df2["X"] == 0].reset_index(drop=True)
untreated_Z = untreated2[["Z1", "Z2"]].values

effects, match_dist, match_idx = [], [], []
for _, row in treated2.iterrows():
    t_point = row[["Z1", "Z2"]].values
    # Distance from this treated row to EVERY untreated row
    dists = np.array([mahalanobis(t_point, u_point, VI) for u_point in untreated_Z])
    best = dists.argmin()                       # nearest untreated row (with replacement)
    effects.append(row["Y"] - untreated2.loc[best, "Y"])
    match_dist.append(dists[best])
    match_idx.append(best)                      # remember WHICH untreated row was the match

treated2["effect"] = effects
treated2["match_dist"] = match_dist
treated2["match_idx"] = match_idx

print(f"Matched ATE: {np.mean(effects):.3f}")

Matched ATE: 3.438


## Question 4: The treated row with the least common support

**Common support** means a treated row has untreated rows that look like it, so its counterfactual is believable.
If even the *nearest* untreated row is far away, the comparison is weak: we're comparing that treated row to
someone quite different.

**Step 1: find the worst-supported treated row.** We stored each treated row's distance to its nearest untreated match
(`match_dist`) in Question 3. The row with the **largest** of these distances has the least common support.

**Step 2: report its nearest values.** The question asks for the *nearest* Z1 and Z2 values **of** that treated item.
That means the Z1 and Z2 of the untreated row it was matched to (its nearest neighbour), not the treated row's own values.
That matched row is the best counterfactual available, and how far it sits from the treated row shows why the support is weak.

In practice, rows like this are often trimmed, because their estimated effect relies on a poor match.

In [8]:
worst = treated2.loc[treated2["match_dist"].idxmax()]
nearest = untreated2.loc[worst["match_idx"]]

print(f"Treated row with least common support: Z1 = {worst['Z1']:.2f}, Z2 = {worst['Z2']:.2f}")
print(f"Its nearest untreated match (ANSWER):  Z1 = {nearest['Z1']:.2f}, Z2 = {nearest['Z2']:.2f}")
print(f"Mahalanobis distance between them:     {worst['match_dist']:.3f}")

Treated row with least common support: Z1 = 2.70, Z2 = 0.54
Its nearest untreated match (ANSWER):  Z1 = 1.52, Z2 = -1.28
Mahalanobis distance between them:     1.383
